In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import sys
import os

# The user specified the path: My Drive/EPA/MLaBP
# Construct the full absolute path within the mounted Drive
notebook_dir_in_drive = '/content/drive/MyDrive/EPA/MLaBP'

# Add this directory to the beginning of sys.path if it's not already there
if notebook_dir_in_drive not in sys.path:
    sys.path.insert(0, notebook_dir_in_drive)
    print(f"Added user-specified directory to sys.path: {notebook_dir_in_drive}")
else:
    print(f"User-specified directory already in sys.path: {notebook_dir_in_drive}")

from tokenizer import CharachterLevelTokenizer, TiktokenTokenizer, MinbpeTokenizer

Added user-specified directory to sys.path: /content/drive/MyDrive/EPA/MLaBP


In [ ]:
base_params = {
    'batch_size': 64,
    'block_size': 128,
    'max_iters': 3000,
    'eval_interval': 300,
    'learning_rate': 1e-2,
    'eval_iters': 200
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)

In [8]:
print(device)

cuda


In [9]:
def get_batch(data, params, device):
    ix = torch.randint(len(data) - params['block_size'], (params['batch_size'],))
    x = torch.stack([data[i:i+params['block_size']] for i in ix])
    y = torch.stack([data[i+1:i+params['block_size']+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [10]:
@torch.no_grad()
def estimate_loss(model, data, params, device):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(params['eval_iters'])
        for k in range(params['eval_iters']):
            X, Y = get_batch(data[split], params, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [11]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [ ]:
import time
from tqdm import tqdm

def run_experiment(tokenizer, name, text, cache_name):
    # --- Encoding (cached) ---
    cache_file = os.path.join(cache_dir, f'{cache_name}.pt')

    if os.path.exists(cache_file):
        print(f"[cache] Loading encoded tokens from {cache_file}")
        payload = torch.load(cache_file)
        encoded = payload['encoded']
        vocab_size = payload['vocab_size']
    else:
        print(f"[cache] No cache found, encoding with {name} tokenizer...")
        encoded = torch.tensor(tokenizer.encode(text), dtype=torch.long)
        vocab_size = len(tokenizer.vocab)
        torch.save({'encoded': encoded, 'vocab_size': vocab_size}, cache_file)
        print(f"[cache] Saved to {cache_file}")

    params = {**base_params, 'vocab_size': vocab_size}

    # Train/val split
    n = int(0.9 * len(encoded))
    data = {'train': encoded[:n], 'val': encoded[n:]}

    # Model + optimizer
    torch.manual_seed(42)
    model = BigramLanguageModel(vocab_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'])

    print(f"\n=== {name} | vocab_size={vocab_size} | tokens={len(encoded)} ===")
    history = []
    t0 = time.time()
    pbar = tqdm(range(params['max_iters'] + 1), desc=name)
    for iter in pbar:
        if iter % params['eval_interval'] == 0:
            losses = estimate_loss(model, data, params, device)
            pbar.set_postfix(train=f"{losses['train']:.4f}", val=f"{losses['val']:.4f}")
            tqdm.write(f"  step {iter:4d}: train {losses['train']:.4f}  val {losses['val']:.4f}")
            history.append({'step': iter, 'train': losses['train'].item(), 'val': losses['val'].item()})
        if iter < params['max_iters']:
            xb, yb = get_batch(data['train'], params, device)
            _, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    t1 = time.time()

    return model, history, t1 - t0

In [ ]:
def compute_metrics(history, vocab_size):
    log_v = math.log(vocab_size)
    enriched = []
    for h in history:
        enriched.append({
            'step': h['step'],
            'train_loss': h['train'],
            'val_loss': h['val'],
            'train_ppl': math.exp(h['train']),
            'val_ppl': math.exp(h['val']),
            'train_norm_loss': h['train'] / log_v,
            'val_norm_loss': h['val'] / log_v,
            'train_norm_ppl': math.exp(h['train'] / log_v),
            'val_norm_ppl': math.exp(h['val'] / log_v),
        })
    return enriched

In [ ]:
import json
import math
import matplotlib.pyplot as plt
from datetime import datetime
from huggingface_hub import hf_hub_download

MINBPE_MAX_CHARS = 100000
DATASETS = ['shakespeare', 'text8']
TOKENIZER_NAMES = ["CharacterLevel", "Tiktoken (cl100k)", "MinBPE"]

# Tiktoken exceeds GPU memory on text8 — skip it for that dataset
SKIP = {
    'text8': {'Tiktoken (cl100k)'},
}

cache_dir = os.path.join(notebook_dir_in_drive, 'data', 'cache')
cache_keys = ['char', 'tiktoken', 'minbpe']

def plot_metric(all_metrics, metric_train, metric_val, title, ylabel, save_path):
    fig, ax = plt.subplots(figsize=(10, 5))
    for metrics, name in zip(all_metrics, TOKENIZER_NAMES):
        if metrics is None:
            continue
        steps = [h['step'] for h in metrics]
        ax.plot(steps, [h[metric_train] for h in metrics], label=f'{name} train')
        ax.plot(steps, [h[metric_val]   for h in metrics], label=f'{name} val', linestyle='--')
    ax.set_title(title); ax.set_xlabel('step'); ax.set_ylabel(ylabel); ax.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

all_results = {}

for DATASET in DATASETS:
    print(f"\n{'='*60}\n  DATASET: {DATASET.upper()}\n{'='*60}")

    skip_set = SKIP.get(DATASET, set())

    # --- Load text ---
    if DATASET == "shakespeare":
        with open('/content/drive/MyDrive/EPA/MLaBP/data/input.txt', 'r', encoding='utf-8') as f:
            text = f.read()
    elif DATASET == "text8":
        sentences_path = hf_hub_download(repo_id="roshbeed/text8-dataset", filename="text8_sentences.json")
        with open(sentences_path, 'r') as f:
            _data = json.load(f)
        text = ' '.join(_data['sentences'])

    print(f"Length: {len(text):,} chars | preview: {text[:80]!r}")

    # --- Tokenizers ---
    tokenizers = [
        CharachterLevelTokenizer(text),
        TiktokenTokenizer(text),
        MinbpeTokenizer(text, max_chars=MINBPE_MAX_CHARS),
    ]
    for tok, name in zip(tokenizers, TOKENIZER_NAMES):
        print(f"{name} vocab: {len(tok.vocab)}")

    # --- Run experiments ---
    results = []
    for tok, name, key in zip(tokenizers, TOKENIZER_NAMES, cache_keys):
        if name in skip_set:
            print(f"\n[SKIP] {name} on {DATASET} (exceeds GPU memory)")
            results.append(None)
        else:
            results.append(run_experiment(tok, name, text, f"{key}_{DATASET}"))

    models    = [r[0] if r is not None else None for r in results]
    histories = [r[1] if r is not None else None for r in results]
    times     = [r[2] if r is not None else None for r in results]

    # --- Compute derived metrics ---
    all_metrics = [
        compute_metrics(h, len(tok.vocab)) if h is not None else None
        for h, tok in zip(histories, tokenizers)
    ]

    experiments = list(zip(TOKENIZER_NAMES, all_metrics, [len(t.vocab) for t in tokenizers], times))

    # --- Summary table ---
    header = f"{'Tokenizer':<20} {'Vocab':>6}  {'TrainLoss':>9} {'ValLoss':>8}  {'TrainPPL':>9} {'ValPPL':>8}  {'NormValLoss':>12} {'NormValPPL':>11}  {'Time(s)':>8}"
    sep = "-" * len(header)
    print(f"\n{sep}\n{header}\n{sep}")

    summary_rows = []
    for name, metrics, vocab_size, train_time in experiments:
        if metrics is None:
            print(f"{name:<20} {'SKIPPED':>6}")
            continue
        last = metrics[-1]
        print(
            f"{name:<20} {vocab_size:>6}  "
            f"{last['train_loss']:>9.4f} {last['val_loss']:>8.4f}  "
            f"{last['train_ppl']:>9.2f} {last['val_ppl']:>8.2f}  "
            f"{last['val_norm_loss']:>12.4f} {last['val_norm_ppl']:>11.4f}  "
            f"{train_time:>8.1f}"
        )
        summary_rows.append({
            'tokenizer': name, 'vocab_size': vocab_size,
            'train_loss': last['train_loss'], 'val_loss': last['val_loss'],
            'train_ppl': last['train_ppl'], 'val_ppl': last['val_ppl'],
            'train_norm_loss': last['train_norm_loss'], 'val_norm_loss': last['val_norm_loss'],
            'train_norm_ppl': last['train_norm_ppl'], 'val_norm_ppl': last['val_norm_ppl'],
            'training_time_s': train_time,
        })
    print(sep)

    # --- Run folder ---
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = os.path.join(notebook_dir_in_drive, f'models/bigram/{DATASET}_{timestamp}')
    os.makedirs(save_dir, exist_ok=True)
    print(f"\nRun folder: {save_dir}")

    # --- Plots ---
    active_histories = [(h, n) for h, n in zip(histories, TOKENIZER_NAMES) if h is not None]
    fig, axes = plt.subplots(1, len(active_histories), figsize=(6 * len(active_histories), 4), sharey=False)
    if len(active_histories) == 1:
        axes = [axes]
    for ax, (history, name) in zip(axes, active_histories):
        steps = [h['step'] for h in history]
        ax.plot(steps, [h['train'] for h in history], label='train')
        ax.plot(steps, [h['val']   for h in history], label='val', linestyle='--')
        ax.set_title(name); ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()
    plt.suptitle(f'Bigram Model [{DATASET}]: Loss by Tokenizer')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'plot_loss_curves.png'))
    plt.show()

    plot_metric(all_metrics, 'train_ppl',       'val_ppl',
                f'Bigram Model [{DATASET}]: Perplexity by Tokenizer', 'PPL',
                os.path.join(save_dir, 'plot_ppl_curves.png'))
    plot_metric(all_metrics, 'train_norm_loss',  'val_norm_loss',
                f'Bigram Model [{DATASET}]: Normalized Loss (loss / log|V|) by Tokenizer', 'Normalized Loss',
                os.path.join(save_dir, 'plot_normalized_loss.png'))
    plot_metric(all_metrics, 'train_norm_ppl',   'val_norm_ppl',
                f'Bigram Model [{DATASET}]: Normalized PPL by Tokenizer', 'Normalized PPL',
                os.path.join(save_dir, 'plot_normalized_ppl.png'))

    # --- Text generation samples ---
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    for model, tok, name in zip(models, tokenizers, TOKENIZER_NAMES):
        if model is None:
            print(f"\n=== [{DATASET}] {name} generation: SKIPPED ===")
            continue
        print(f"\n=== [{DATASET}] {name} generation ===")
        print(tok.decode(model.generate(context, max_new_tokens=200)[0].tolist()))

    # --- Save ---
    for model, key in zip(models, cache_keys):
        if model is None:
            continue
        torch.save(model.state_dict(), os.path.join(save_dir, f'{key}_model.pt'))
    with open(os.path.join(save_dir, 'base_params.json'), 'w') as f:
        json.dump(base_params, f, indent=4)
    with open(os.path.join(save_dir, 'summary.json'), 'w') as f:
        json.dump(summary_rows, f, indent=4)
    print(f"All outputs saved to {save_dir}/")

    all_results[DATASET] = {'summary': summary_rows, 'metrics': all_metrics}

print(f"\n{'='*60}\n  ALL DATASETS COMPLETE\n{'='*60}")